In [ ]:
from pyspark.sql import SparkSession
from google.cloud import storage
import json

In [ ]:
hadoop_version = (
    spark.sparkContext
    ._jvm
    .org.apache.hadoop.util.VersionInfo
    .getVersion()
)

print(hadoop_version)

In [ ]:
!mkdir -p "$HOME/spark-jars"

!curl -fL \
  "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar" \
  -o "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"

In [ ]:
!ls -lh "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"

In [ ]:
import os
from pyspark.sql import SparkSession

gcs_connector_jar = os.path.expanduser(
    "~/spark-jars/gcs-connector-hadoop3-latest.jar"
)

if not os.path.isfile(gcs_connector_jar):
    raise FileNotFoundError(
        f"Nie znaleziono konektora: {gcs_connector_jar}"
    )

spark = (
    SparkSession.builder
    .appName("development_for_silver_layer")
    .config(
        "spark.jars",
        gcs_connector_jar
    )
    .config(
        "spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
    )
    .config(
        "spark.hadoop.fs.AbstractFileSystem.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS"
    )
    .getOrCreate()
)

In [ ]:
spark = SparkSession \
        .builder\
        .appName("development_for_silver_layer") \
        .getOrCreate()

In [ ]:
spark.version

In [ ]:
storage_client = storage.Client()
bucket = storage_client.get_bucket("project-dev-storage")

In [ ]:
blob = bucket.get_blob("company_forms/sec_metadata_2")
data_string = blob.download_as_string()

In [ ]:
fil = json.loads(data_string)

#df = spark.createDataFrame(fil)
#df

In [ ]:
print(
    spark.sparkContext
    ._jsc
    .hadoopConfiguration()
    .get("fs.gs.impl")
)

In [ ]:
df = (
    spark.read
    .format("json")
    .option("multiline", "true")
    .load("gs://project-dev-storage/company_forms/sec_metadata_2")
    .limit(1)
)



In [ ]:
df.select(
    "cik",
    "name",
    "entityType",
    "sic",
    "category"
).show(1, truncate=False)